In [ ]:
# =========================
# Colab Script (Recommended)
# - Baseline evaluation first (no HP tuning yet)
# - Feature-count hyperparameter handled via top-k EDGE selection (recommended)
#   * O  : original features only (fixed)
#   * F  : edge features only (top-k by |weight|)
#   * OF : original + top-k edge features (original always included)
#
# Metrics computed (5): F1, AUROC, AUPRC, Brier, ECE
# Model selection focus: AUROC, AUPRC (you can rank by these)
#
# Outputs: 3 CSVs (O / F / OF) with columns:
#   SET, DAG, MODEL, K_EDGE, AUROC, AUPRC, F1, Brier, ECE
# =========================

!pip -q install lightgbm xgboost

import os
import re
import json
import math
import random
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

from typing import Dict, List, Tuple, Optional

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb
import lightgbm as lgb

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# =========================
# Config
# =========================
RANDOM_STATE = 42

TEST_SIZE = 0.2
VAL_SIZE = 0.2  # val portion within trainval

ECE_BINS = 15

# FFMLP baseline config
BATCH_SIZE = 2048
EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 5

# ---- Feature-count hyperparameter (recommended) ----
# We tune ONLY the number of EDGE features via top-k by |weight|.
# O  : ignore K_EDGE (set to 0)
# F  : use K_EDGE edge features
# OF : original + K_EDGE edge features
K_EDGE_CANDIDATES = [5, 10, 20, 40, "ALL"]  # you can edit

# =========================
# Paths (Google Drive)
# =========================
BASE_DIR = "/content/drive/MyDrive/bank_failure_prediction"  # 필요하면 하위 폴더로 수정

DATA_ORIGINAL = f"{BASE_DIR}/training_data_normalized.csv"

DATA_DAG = {
    "NOTEARS": f"{BASE_DIR}/training_data_normalized_with_edge_features_NOTEARS.csv",
    "PC":      f"{BASE_DIR}/training_data_normalized_with_edge_features_PC.csv",
    "GES":     f"{BASE_DIR}/training_data_normalized_with_edge_features_GES.csv",
    "GOLEM":   f"{BASE_DIR}/training_data_normalized_with_edge_features_GOLEM.csv",
}

GEXF_PATHS = {
    "NOTEARS": f"{BASE_DIR}/graph_NOTEARS.gexf",
    "PC":      f"{BASE_DIR}/graph_PC.gexf",
    "GES":     f"{BASE_DIR}/graph_GES.gexf",
    "GOLEM":   f"{BASE_DIR}/graph_GOLEM.gexf",
}

OUT_DIR = f"{BASE_DIR}/results_tables"
os.makedirs(OUT_DIR, exist_ok=True)


# =========================
# Seed
# =========================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)


# =========================
# Metrics
# =========================
def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 15) -> float:
    y_true = y_true.astype(int)
    y_prob = np.clip(y_prob, 0.0, 1.0)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(y_true)

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi) if i < n_bins - 1 else (y_prob >= lo) & (y_prob <= hi)
        if not np.any(mask):
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return float(ece)

def brier_score(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = y_true.astype(float)
    y_prob = np.clip(y_prob, 0.0, 1.0)
    return float(np.mean((y_prob - y_true) ** 2))

def best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray, n_grid: int = 101) -> float:
    thresholds = np.linspace(0.0, 1.0, n_grid)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)

def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "AUROC": float(roc_auc_score(y_true, y_prob)),
        "AUPRC": float(average_precision_score(y_true, y_prob)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Brier": brier_score(y_true, y_prob),
        "ECE": expected_calibration_error(y_true, y_prob, n_bins=ECE_BINS),
    }


# =========================
# Data utils
# =========================
def detect_target_col(df: pd.DataFrame) -> str:
    candidates = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Target column not found. Tried {candidates}")

def split_data(X: np.ndarray, y: np.ndarray):
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_trainval
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

def to_numpy(df: pd.DataFrame, cols: List[str]) -> np.ndarray:
    return df[cols].to_numpy(dtype=np.float32)


# =========================
# Build top-k edge feature list by |weight|
# =========================
import networkx as nx

def get_weight(attrs: dict) -> Optional[float]:
    w = attrs.get("weight", None)
    if w is None:
        w = attrs.get("value", None)
    if w is None:
        return None
    try:
        return float(w)
    except:
        return None

def load_edge_ranking_by_abs_weight(alg: str, gexf_path: str, df_alg: pd.DataFrame) -> List[str]:
    """
    Return edge feature column names sorted by |weight| desc.
    Assumes column naming: edge_<ALG>__<u>__<v>
    """
    G = nx.read_gexf(gexf_path)
    if not isinstance(G, (nx.DiGraph, nx.MultiDiGraph)):
        G = nx.DiGraph(G)

    edge_items = []
    for u, v, attrs in G.edges(data=True):
        w = get_weight(attrs)
        if w is None:
            continue
        col = f"edge_{alg}__{str(u)}__{str(v)}"
        # handle possible dup suffixes
        if col not in df_alg.columns:
            matches = [c for c in df_alg.columns if c.startswith(col)]
            if matches:
                col = matches[0]
            else:
                # fallback (older naming)
                col2 = f"edge_{str(u)}__{str(v)}"
                if col2 in df_alg.columns:
                    col = col2
                else:
                    matches2 = [c for c in df_alg.columns if c.startswith(col2)]
                    if matches2:
                        col = matches2[0]
                    else:
                        continue
        edge_items.append((abs(w), col))

    edge_items.sort(key=lambda x: x[0], reverse=True)
    ranked_cols = [c for _, c in edge_items]
    # de-dup in case
    seen = set()
    out = []
    for c in ranked_cols:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out


# =========================
# Models
# =========================
def fit_predict_logit(X_train, y_train, X_val, y_val, X_test):
    clf = LogisticRegression(max_iter=2000, solver="lbfgs")
    clf.fit(X_train, y_train)
    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob)
    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr

def fit_predict_rf(X_train, y_train, X_val, y_val, X_test):
    clf = RandomForestClassifier(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced_subsample",
    )
    clf.fit(X_train, y_train)
    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob)
    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr

def fit_predict_xgb(X_train, y_train, X_val, y_val, X_test):
    clf = xgb.XGBClassifier(
        # GPU 사용 (XGBoost 3.1+ 방식)
        tree_method="hist",
        device="cuda",

        n_estimators=800,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    clf.fit(X_train, y_train)

    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob)

    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr


def fit_predict_lgbm(X_train, y_train, X_val, y_val, X_test):
    clf = lgb.LGBMClassifier(
        device="gpu",
        gpu_platform_id=0,
        gpu_device_id=0,
        n_estimators=2000,
        learning_rate=0.02,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
    )
    clf.fit(X_train, y_train)
    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob)
    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr


# =========================
# FFMLP (PyTorch)
# =========================
class FFMLP(nn.Module):
    def __init__(self, in_dim: int, hidden: List[int] = [128, 64], dropout: float = 0.1):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

@torch.no_grad()
def predict_proba_mlp(model: nn.Module, X: np.ndarray, device: str) -> np.ndarray:
    model.eval()
    dl = DataLoader(TensorDataset(torch.tensor(X)), batch_size=4096, shuffle=False)
    probs = []
    for (xb,) in dl:
        xb = xb.to(device)
        logits = model(xb)
        p = torch.sigmoid(logits).detach().cpu().numpy()
        probs.append(p)
    return np.concatenate(probs, axis=0)

def train_mlp(X_train, y_train, X_val, y_val) -> Tuple[FFMLP, float]:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = FFMLP(in_dim=X_train.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=4096, shuffle=False)

    best_val = float("inf")
    best_state = None
    bad = 0

    for _ in range(EPOCHS):
        model.train()
        for xb, yb in train_dl:
            xb = xb.to(device)
            yb = yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

        model.eval()
        vlosses = []
        with torch.no_grad():
            for xb, yb in val_dl:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                vlosses.append(loss_fn(logits, yb).item())
        v = float(np.mean(vlosses))

        if v < best_val - 1e-6:
            best_val = v
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= EARLY_STOPPING_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    y_val_prob = predict_proba_mlp(model, X_val, device=device)
    thr = best_f1_threshold(y_val, y_val_prob)
    return model, thr


# =========================
# Feature sets (recommended)
# =========================
def get_original_cols(df_original: pd.DataFrame, target_col: str) -> List[str]:
    return [c for c in df_original.columns if c != target_col and not c.startswith("edge_")]

def pick_k_edge_cols(edge_ranked_cols: List[str], k_edge) -> List[str]:
    if k_edge == "ALL":
        return edge_ranked_cols
    k_edge = int(k_edge)
    return edge_ranked_cols[:min(k_edge, len(edge_ranked_cols))]


# =========================
# Run baseline evaluations
# =========================
df_original = pd.read_csv(DATA_ORIGINAL, low_memory=False)
target_col = detect_target_col(df_original)
y = df_original[target_col].to_numpy(dtype=np.int64)

orig_cols = get_original_cols(df_original, target_col)

MODELS = ["LightGBM", "XGBoost", "FFMLP", "Logit", "RF"]

rows_O, rows_F, rows_OF = [], [], []

for alg, dag_csv in DATA_DAG.items():
    df_alg = pd.read_csv(dag_csv, low_memory=False)

    # edge ranking by |w|
    edge_ranked_cols = load_edge_ranking_by_abs_weight(alg, GEXF_PATHS[alg], df_alg)

    # ---------- O (fixed) ----------
    X = to_numpy(df_original, orig_cols)
    X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

    # Evaluate models
    # LGBM
    prob, thr = fit_predict_lgbm(X_train, y_train, X_val, y_val, X_test)
    met = compute_metrics(y_test, prob, thr)
    rows_O.append({"SET":"O","DAG":alg,"MODEL":"LightGBM","K_EDGE":0,**met})

    # XGB
    prob, thr = fit_predict_xgb(X_train, y_train, X_val, y_val, X_test)
    met = compute_metrics(y_test, prob, thr)
    rows_O.append({"SET":"O","DAG":alg,"MODEL":"XGBoost","K_EDGE":0,**met})

    # FFMLP
    model_mlp, thr = train_mlp(
        X_train.astype(np.float32), y_train.astype(np.float32),
        X_val.astype(np.float32), y_val.astype(np.float32)
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
    prob = predict_proba_mlp(model_mlp, X_test.astype(np.float32), device=device)
    met = compute_metrics(y_test, prob, thr)
    rows_O.append({"SET":"O","DAG":alg,"MODEL":"FFMLP","K_EDGE":0,**met})

    # Logit
    prob, thr = fit_predict_logit(X_train, y_train, X_val, y_val, X_test)
    met = compute_metrics(y_test, prob, thr)
    rows_O.append({"SET":"O","DAG":alg,"MODEL":"Logit","K_EDGE":0,**met})

    # RF
    prob, thr = fit_predict_rf(X_train, y_train, X_val, y_val, X_test)
    met = compute_metrics(y_test, prob, thr)
    rows_O.append({"SET":"O","DAG":alg,"MODEL":"RF","K_EDGE":0,**met})

    # ---------- F / OF (K_EDGE candidates) ----------
    for k_edge in K_EDGE_CANDIDATES:
        edge_cols_used = pick_k_edge_cols(edge_ranked_cols, k_edge)

        # F: edge only
        if len(edge_cols_used) > 0:
            Xf = to_numpy(df_alg, edge_cols_used)
            X_train, X_val, X_test, y_train, y_val, y_test = split_data(Xf, y)

            # LGBM
            prob, thr = fit_predict_lgbm(X_train, y_train, X_val, y_val, X_test)
            met = compute_metrics(y_test, prob, thr)
            rows_F.append({"SET":"F","DAG":alg,"MODEL":"LightGBM","K_EDGE":("ALL" if k_edge=="ALL" else int(k_edge)),**met})

            # XGB
            prob, thr = fit_predict_xgb(X_train, y_train, X_val, y_val, X_test)
            met = compute_metrics(y_test, prob, thr)
            rows_F.append({"SET":"F","DAG":alg,"MODEL":"XGBoost","K_EDGE":("ALL" if k_edge=="ALL" else int(k_edge)),**met})

            # FFMLP
            model_mlp, thr = train_mlp(
                X_train.astype(np.float32), y_train.astype(np.float32),
                X_val.astype(np.float32), y_val.astype(np.float32)
            )
            prob = predict_proba_mlp(model_mlp, X_test.astype(np.float32), device=device)
            met = compute_metrics(y_test, prob, thr)
            rows_F.append({"SET":"F","DAG":alg,"MODEL":"FFMLP","K_EDGE":("ALL" if k_edge=="ALL" else int(k_edge)),**met})

            # Logit
            prob, thr = fit_predict_logit(X_train, y_train, X_val, y_val, X_test)
            met = compute_metrics(y_test, prob, thr)
            rows_F.append({"SET":"F","DAG":alg,"MODEL":"Logit","K_EDGE":("ALL" if k_edge=="ALL" else int(k_edge)),**met})

            # RF
            prob, thr = fit_predict_rf(X_train, y_train, X_val, y_val, X_test)
            met = compute_metrics(y_test, prob, thr)
            rows_F.append({"SET":"F","DAG":alg,"MODEL":"RF","K_EDGE":("ALL" if k_edge=="ALL" else int(k_edge)),**met})

        # OF: original + edge (original always included)
        of_cols = orig_cols + edge_cols_used
        Xof = to_numpy(df_alg, of_cols)
        X_train, X_val, X_test, y_train, y_val, y_test = split_data(Xof, y)

        # LGBM
        prob, thr = fit_predict_lgbm(X_train, y_train, X_val, y_val, X_test)
        met = compute_metrics(y_test, prob, thr)
        rows_OF.append({"SET":"OF","DAG":alg,"MODEL":"LightGBM","K_EDGE":("ALL" if k_edge=="ALL" else int(k_edge)),**met})

        # XGB
        prob, thr = fit_predict_xgb(X_train, y_train, X_val, y_val, X_test)
        met = compute_metrics(y_test, prob, thr)
        rows_OF.append({"SET":"OF","DAG":alg,"MODEL":"XGBoost","K_EDGE":("ALL" if k_edge=="ALL" else int(k_edge)),**met})

        # FFMLP
        model_mlp, thr = train_mlp(
            X_train.astype(np.float32), y_train.astype(np.float32),
            X_val.astype(np.float32), y_val.astype(np.float32)
        )
        prob = predict_proba_mlp(model_mlp, X_test.astype(np.float32), device=device)
        met = compute_metrics(y_test, prob, thr)
        rows_OF.append({"SET":"OF","DAG":alg,"MODEL":"FFMLP","K_EDGE":("ALL" if k_edge=="ALL" else int(k_edge)),**met})

        # Logit
        prob, thr = fit_predict_logit(X_train, y_train, X_val, y_val, X_test)
        met = compute_metrics(y_test, prob, thr)
        rows_OF.append({"SET":"OF","DAG":alg,"MODEL":"Logit","K_EDGE":("ALL" if k_edge=="ALL" else int(k_edge)),**met})

        # RF
        prob, thr = fit_predict_rf(X_train, y_train, X_val, y_val, X_test)
        met = compute_metrics(y_test, prob, thr)
        rows_OF.append({"SET":"OF","DAG":alg,"MODEL":"RF","K_EDGE":("ALL" if k_edge=="ALL" else int(k_edge)),**met})


# =========================
# Save CSVs (focus: AUROC, AUPRC first, but keep all metrics)
# =========================
def finalize_table(rows: List[dict], set_name: str) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    # Ensure column order
    df = df[["SET","DAG","MODEL","K_EDGE","AUROC","AUPRC","F1","Brier","ECE"]]
    # sort by DAG then by AUROC/AUPRC (descending), then model name
    df = df.sort_values(["DAG","AUROC","AUPRC","MODEL"], ascending=[True, False, False, True]).reset_index(drop=True)
    df["SET"] = set_name
    return df

tbl_O  = finalize_table(rows_O, "O")
tbl_F  = finalize_table(rows_F, "F")
tbl_OF = finalize_table(rows_OF, "OF")

out_o  = os.path.join(OUT_DIR, "results_O.csv")
out_f  = os.path.join(OUT_DIR, "results_F.csv")
out_of = os.path.join(OUT_DIR, "results_OF.csv")

tbl_O.to_csv(out_o, index=False)
tbl_F.to_csv(out_f, index=False)
tbl_OF.to_csv(out_of, index=False)

print("[DONE] saved:")
print(out_o)
print(out_f)
print(out_of)

# Quick peek
display(tbl_O.head(10))
display(tbl_F.head(10))
display(tbl_OF.head(10))
